
# Laguna XS.2 causal expert search — Kaggle T4×2

This notebook replaces the previous Transformers loader with **vLLM quantized execution**.

## What it does

1. Loads `poolside/Laguna-XS.2-INT4` with TP=2 without CPU offload/swap.
2. Instruments vLLM **after the real MoE router selects top-k experts**.
3. Supports fixed-routing causal ablation:
   - zero a whole routed MoE layer;
   - zero chosen experts without allowing expert #9 to replace them.
4. Uses teacher-forced reference NLL to measure target vs control damage.
5. Runs a **causal-first** search:
   - 39 layer interventions;
   - hierarchical expert group testing;
   - exact individual expert validation.
6. Optionally captures routing only for comparison with causal importance.

The routing statistics are **not used as the main selector**.

> Start from a fresh Kaggle session with **T4 ×2** and Internet enabled.


## 1. Install

In [ ]:

# Run in a fresh kernel before importing vLLM elsewhere.
%pip -q install -U "vllm>=0.25.0" "transformers>=5.7.0" pandas psutil pyarrow


## 2. Hardware / RAM check

In [ ]:

import os, json, time, shutil, subprocess, glob, re
from pathlib import Path
import psutil

ram = psutil.virtual_memory()
print(f"RAM total: {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")
subprocess.run(["nvidia-smi"], check=False)

if ram.available / 2**30 < 18:
    raise RuntimeError("Restart Kaggle: less than 18 GiB host RAM is currently available.")



## 3. Patch vLLM's modular MoE runner

We patch immediately after vLLM's actual:

```python
topk_weights, topk_ids = self.router.select_experts(...)
```

That means the IDs already include Laguna's true routing logic.

The patch is controlled by `/kaggle/working/laguna_xs2_causal/control.json`.

It can:

- capture actual top-k IDs/weights;
- zero selected expert weights;
- zero all routed experts in one layer;
- optionally renormalize the surviving selected weights.

The original selected top-k IDs remain fixed during expert ablation.


### Required XS.2 INT4 compatibility fix

Poolside's INT4 checkpoint mixes INT4 and **group-quantized INT8** experts.  
Some packaged vLLM builds still contain an obsolete assertion in the WNA16 MoE path:

```python
assert self.group_size == -1
```

Upstream vLLM PR **#47154** removed exactly that assertion because Marlin supports grouped INT8 (including group size 128), and the old line prevents `poolside/Laguna-XS.2-INT4` from initializing.

This notebook applies that exact one-line upstream compatibility patch **before importing vLLM**, then verifies the assertion is gone.


In [ ]:

import importlib.util
from pathlib import Path
import shutil

spec = importlib.util.find_spec("vllm")
if spec is None or not spec.submodule_search_locations:
    raise RuntimeError("vLLM not installed.")

vllm_root = Path(list(spec.submodule_search_locations)[0])
runner_path = vllm_root / "model_executor/layers/fused_moe/runner/moe_runner.py"
backup_path = runner_path.with_suffix(".py.atlas_backup")

src = runner_path.read_text()
PATCH_MARK = "# === LAGUNA_CAUSAL_ATLAS_PATCH_V1 ==="

print("Patching:", runner_path)

# ------------------------------------------------------------------
# Compatibility patch: vLLM PR #47154
# poolside/Laguna-XS.2-INT4 mixes INT4 with group-quantized INT8
# experts. Some packaged vLLM builds still contain the obsolete
# `assert self.group_size == -1` in the INT8 WNA16 MoE path.
#
# Upstream fix #47154 is exactly to remove that assertion.
# We patch only files under compressed_tensors_moe whose INT8 branch
# contains the exact obsolete assertion, and verify it is gone.
# ------------------------------------------------------------------
ct_moe_dir = (
    vllm_root
    / "model_executor/layers/quantization/compressed_tensors"
    / "compressed_tensors_moe"
)

compat_candidates = [
    ct_moe_dir / "compressed_tensors_moe_wna16.py",
    ct_moe_dir / "compressed_tensors_moe_wna16_marlin.py",
]

compat_touched = []

for compat_path in compat_candidates:
    if not compat_path.exists():
        continue

    txt = compat_path.read_text()

    # Be deliberately narrow: only remove the exact obsolete assertion.
    old = "        assert self.group_size == -1\n"
    if old in txt:
        backup = compat_path.with_suffix(compat_path.suffix + ".pr47154_backup")
        if not backup.exists():
            shutil.copy2(compat_path, backup)

        txt2 = txt.replace(old, "", 1)
        compat_path.write_text(txt2)
        compat_touched.append(str(compat_path))

    # Whether already fixed upstream or just patched, the obsolete assertion
    # must not remain in the relevant file.
    now = compat_path.read_text()
    if "assert self.group_size == -1" in now:
        raise RuntimeError(
            f"PR #47154 compatibility patch did not remove the obsolete "
            f"grouped-INT8 assertion from {compat_path}"
        )

print("PR #47154 compatibility:", "patched" if compat_touched else "already present")
for p in compat_touched:
    print("  patched:", p)

if PATCH_MARK not in src:
    if not backup_path.exists():
        shutil.copy2(runner_path, backup_path)

    import_anchor = "from collections.abc import Callable, Iterable\n"
    if import_anchor not in src:
        raise RuntimeError("Unexpected vLLM source layout at imports.")

    src = src.replace(
        import_anchor,
        import_anchor + "import json as _atlas_json\nimport os as _atlas_os\n",
        1,
    )

    select_block = '''            topk_weights, topk_ids = self.router.select_experts(
                hidden_states=hidden_states,
                router_logits=router_logits,
                topk_indices_dtype=self._quant_method.topk_indices_dtype,
                input_ids=input_ids,
            )
'''

    if select_block not in src:
        idx = src.find("topk_weights, topk_ids")
        if idx >= 0:
            print(src[max(0, idx-400):idx+1200])
        raise RuntimeError(
            "Expected select_experts block not found. "
            "vLLM changed; refusing to patch automatically."
        )

    patch = r'''
            # === LAGUNA_CAUSAL_ATLAS_PATCH_V1 ===
            _atlas_control_path = _atlas_os.environ.get(
                "LAGUNA_ATLAS_CONTROL",
                "/kaggle/working/laguna_xs2_causal/control.json",
            )
            if _atlas_os.path.exists(_atlas_control_path):
                try:
                    with open(_atlas_control_path, "r") as _atlas_f:
                        _atlas_ctl = _atlas_json.load(_atlas_f)
                except Exception:
                    _atlas_ctl = {}

                _atlas_zero_layers = set(_atlas_ctl.get("zero_layers", []))
                _atlas_targets = _atlas_ctl.get("ablate", {}).get(self.layer_name, [])

                # Fixed-routing interventions: do not re-run top-k.
                if self.layer_name in _atlas_zero_layers:
                    topk_weights = torch.zeros_like(topk_weights)
                elif _atlas_targets:
                    _atlas_t = torch.tensor(
                        _atlas_targets,
                        device=topk_ids.device,
                        dtype=topk_ids.dtype,
                    )
                    _atlas_keep = ~torch.isin(topk_ids, _atlas_t)
                    topk_weights = topk_weights * _atlas_keep.to(topk_weights.dtype)

                    if bool(_atlas_ctl.get("renormalize", False)):
                        _atlas_den = topk_weights.sum(dim=-1, keepdim=True)
                        topk_weights = torch.where(
                            _atlas_den > 0,
                            topk_weights / _atlas_den.clamp_min(1e-12),
                            topk_weights,
                        )

                # Optional actual-routing capture.
                if bool(_atlas_ctl.get("capture", False)):
                    try:
                        _atlas_rank = (
                            torch.distributed.get_rank()
                            if torch.distributed.is_available()
                            and torch.distributed.is_initialized()
                            else 0
                        )
                        _atlas_ids = topk_ids.detach().reshape(-1).to(torch.int64)
                        _atlas_ws = topk_weights.detach().reshape(-1).float()
                        _atlas_valid = _atlas_ids >= 0
                        _atlas_ids = _atlas_ids[_atlas_valid]
                        _atlas_ws = _atlas_ws[_atlas_valid]

                        if _atlas_ids.numel() > 0:
                            _atlas_n = int(self.moe_config.num_logical_experts)
                            _atlas_counts = torch.bincount(
                                _atlas_ids, minlength=_atlas_n
                            )
                            _atlas_wsum = torch.zeros(
                                _atlas_n,
                                device=_atlas_ws.device,
                                dtype=torch.float32,
                            )
                            _atlas_wsum.scatter_add_(0, _atlas_ids, _atlas_ws)
                            _atlas_active = torch.nonzero(
                                _atlas_counts > 0, as_tuple=False
                            ).reshape(-1)

                            _atlas_rec = {
                                "layer_name": str(self.layer_name),
                                "rank": int(_atlas_rank),
                                "tokens": int(hidden_states.shape[0]),
                                "counts": {
                                    str(int(i)): int(_atlas_counts[i].item())
                                    for i in _atlas_active
                                },
                                "weight_sums": {
                                    str(int(i)): float(_atlas_wsum[i].item())
                                    for i in _atlas_active
                                },
                            }

                            _atlas_base = _atlas_ctl.get("output_base")
                            if _atlas_base:
                                _atlas_out = str(_atlas_base) + f".rank{int(_atlas_rank)}.jsonl"
                                with open(_atlas_out, "a") as _atlas_f:
                                    _atlas_f.write(_atlas_json.dumps(_atlas_rec) + "\n")
                    except Exception:
                        pass
            # === END LAGUNA_CAUSAL_ATLAS_PATCH_V1 ===
'''

    src = src.replace(select_block, select_block + patch, 1)
    runner_path.write_text(src)

patched = runner_path.read_text()
assert PATCH_MARK in patched
print("Patch active.")



## 4. Load XS.2 with low host-RAM pressure

Important settings:

- `safetensors_load_strategy="lazy"`
- `max_parallel_loading_workers=1`
- `swap_space=0`
- `cpu_offload_gb=0`
- `tensor_parallel_size=2`
- short context / one sequence
- eager mode

Do not add CPU offload if Kaggle RAM is the bottleneck.


**vLLM 0.27.1 note:** `swap_space` was removed from `EngineArgs`, so this notebook does not pass it. The loader cell checks every configured keyword against the installed `LLM` / `EngineArgs` API before allocating the model.


### T4 KV-cache compatibility

Laguna XS.2 INT4 advertises an FP8 KV-cache scheme in the compressed checkpoint.  
Tesla T4 is SM75 and vLLM's Triton attention backend cannot use FP8 KV cache there.

This notebook explicitly sets:

```python
kv_cache_dtype="float16"
```

An explicit KV-cache dtype overrides the checkpoint's FP8 KV-cache scheme. Because float16 KV uses more memory than FP8, the notebook keeps context short (`max_model_len=1536`, and reduce to 1024 if necessary).

`max_parallel_loading_workers` was removed from this notebook because vLLM 0.27.1 reports that it is currently ignored.


In [ ]:

import os, json, time
from pathlib import Path

MODEL_ID = "poolside/Laguna-XS.2-INT4"
WORK = Path("/kaggle/working/laguna_xs2_causal")
WORK.mkdir(parents=True, exist_ok=True)

CONTROL = WORK / "control.json"
os.environ["LAGUNA_ATLAS_CONTROL"] = str(CONTROL)

HF_CACHE = Path("/kaggle/working/hf_models")
HF_CACHE.mkdir(parents=True, exist_ok=True)

# Disable interventions before vLLM warmup.
CONTROL.write_text(json.dumps({
    "capture": False,
    "zero_layers": [],
    "ablate": {},
    "renormalize": False,
}))

from vllm import LLM, SamplingParams
import vllm
import inspect
from dataclasses import fields
from vllm.engine.arg_utils import EngineArgs

print("vLLM:", vllm.__version__)

# vLLM's Python API evolves quickly. Validate our kwargs against the
# installed LLM signature + EngineArgs dataclass before starting a 24GB load.
_llm_direct = set(inspect.signature(LLM.__init__).parameters)
_engine_args = {f.name for f in fields(EngineArgs)}

MODEL_KWARGS = dict(
    model=MODEL_ID,
    tensor_parallel_size=2,
    distributed_executor_backend="mp",
    trust_remote_code=True,

    safetensors_load_strategy="lazy",
    dtype="float16",
    kv_cache_dtype="float16",  # T4/SM75 cannot use Laguna checkpoint FP8 KV cache
    max_model_len=1536,
    max_num_batched_tokens=1536,
    max_num_seqs=1,
    gpu_memory_utilization=0.95,
    cpu_offload_gb=0,

    enforce_eager=True,
    enable_prefix_caching=False,
    download_dir=str(HF_CACHE),
    seed=42,
)

_unsupported = sorted(
    k for k in MODEL_KWARGS
    if k not in _llm_direct and k not in _engine_args
)
if _unsupported:
    raise TypeError(
        f"Unsupported vLLM {vllm.__version__} arguments: {_unsupported}. "
        "Update the notebook config instead of attempting the model load."
    )

print("Validated vLLM kwargs:", sorted(MODEL_KWARGS))
llm = LLM(**MODEL_KWARGS)

print("XS.2 loaded.")


## 5. Smoke generation

In [ ]:

out = llm.generate(
    ["Reply with exactly: atlas ready"],
    SamplingParams(temperature=0.0, max_tokens=12),
    use_tqdm=False,
)
print(out[0].outputs[0].text)


## 6. Dynamic intervention / capture helpers

In [ ]:

import os, json, glob
from pathlib import Path

def set_control(*, capture=False, output_base=None, zero_layers=None,
                ablate=None, renormalize=False):
    payload = {
        "capture": bool(capture),
        "output_base": str(output_base) if output_base is not None else None,
        "zero_layers": list(zero_layers or []),
        "ablate": {str(k): [int(x) for x in v] for k, v in (ablate or {}).items()},
        "renormalize": bool(renormalize),
    }
    tmp = CONTROL.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload))
    os.replace(tmp, CONTROL)

def clear_control():
    set_control()

def remove_capture(base):
    for p in glob.glob(str(base) + ".rank*.jsonl"):
        os.remove(p)

def read_capture(base, rank=0):
    p = Path(str(base) + f".rank{rank}.jsonl")
    if not p.exists():
        return []
    return [json.loads(x) for x in p.read_text().splitlines() if x.strip()]



## 7. Verify actual router capture

This is only a sanity check/diagnostic. It should reveal about 39 MoE layer names.


In [ ]:

base = WORK / "router_smoke"
remove_capture(base)

set_control(capture=True, output_base=base)

_ = llm.generate(
    ["A React flex child overflows horizontally. What CSS issue would you inspect?"],
    SamplingParams(temperature=0.0, max_tokens=1),
    use_tqdm=False,
)

clear_control()

router_rows = read_capture(base, rank=0)
print("records:", len(router_rows))

MOE_LAYERS = sorted(
    set(r["layer_name"] for r in router_rows),
    key=lambda s: int(re.search(r"layers\.(\d+)", s).group(1)),
)

print("MoE layers:", len(MOE_LAYERS))
print(MOE_LAYERS[:5], "...", MOE_LAYERS[-3:])

if not router_rows:
    raise RuntimeError("Router capture failed. Stop before causal testing.")



## 8. Tiny causal target/control set

This is only for pipeline validation. Replace with a much larger matched set for real experiments.


In [ ]:

import pandas as pd

EVAL = pd.DataFrame([
    {"kind":"target",
     "prefix":"Question: Which CSS declaration lets a flex child shrink below its content width?\nAnswer: ",
     "reference":"min-width: 0;"},
    {"kind":"target",
     "prefix":"Question: Which CSS declaration establishes a flex formatting context?\nAnswer: ",
     "reference":"display: flex;"},
    {"kind":"target",
     "prefix":"Question: Which React hook handles local component state?\nAnswer: ",
     "reference":"useState"},
    {"kind":"target",
     "prefix":"Question: Which CSS property clips horizontal overflow?\nAnswer: ",
     "reference":"overflow-x: hidden;"},

    {"kind":"control",
     "prefix":"Question: Which Python keyword yields a value from a generator?\nAnswer: ",
     "reference":"yield"},
    {"kind":"control",
     "prefix":"Question: Which traversal finds shortest paths in an unweighted graph?\nAnswer: ",
     "reference":"BFS"},
    {"kind":"control",
     "prefix":"Question: Which Java keyword declares class inheritance?\nAnswer: ",
     "reference":"extends"},
    {"kind":"control",
     "prefix":"Question: Which SQL keyword removes duplicate SELECT rows?\nAnswer: ",
     "reference":"DISTINCT"},
])

EVAL


## 9. Teacher-forced reference NLL using vLLM prompt logprobs

In [ ]:

from transformers import AutoTokenizer
import numpy as np

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def lp_value(x):
    if x is None:
        return None
    if isinstance(x, (float, int)):
        return float(x)
    if hasattr(x, "logprob"):
        return float(x.logprob)
    if isinstance(x, dict) and "logprob" in x:
        return float(x["logprob"])
    return None

def chosen_lp(position, token_id):
    if position is None:
        return None
    if isinstance(position, dict):
        if token_id in position:
            return lp_value(position[token_id])
        if str(token_id) in position:
            return lp_value(position[str(token_id)])
    try:
        return lp_value(position[token_id])
    except Exception:
        return None

def ref_nll(prefix, reference):
    full = prefix + reference
    prefix_ids = tok.encode(prefix, add_special_tokens=False)
    full_ids = tok.encode(full, add_special_tokens=False)

    # Longest common prefix handles possible tokenizer merge at the boundary.
    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    o = llm.generate(
        [full],
        SamplingParams(
            temperature=0.0,
            max_tokens=1,
            prompt_logprobs=1,
        ),
        use_tqdm=False,
    )[0]

    ids = list(o.prompt_token_ids)
    plps = o.prompt_logprobs
    if plps is None:
        raise RuntimeError("No prompt_logprobs returned.")

    losses = []
    for i in range(max(1, min(start, len(ids)-1)), len(ids)):
        lp = chosen_lp(plps[i], int(ids[i]))
        if lp is not None and np.isfinite(lp):
            losses.append(-lp)

    if not losses:
        raise RuntimeError("Could not extract answer-token prompt logprobs.")

    return float(np.mean(losses))

def score_set(df):
    vals = []
    for r in df.itertuples(index=False):
        vals.append({
            "kind": r.kind,
            "nll": ref_nll(r.prefix, r.reference),
        })
    return pd.DataFrame(vals)

clear_control()
BASE = score_set(EVAL)
BASE_T = BASE[BASE.kind=="target"].nll.mean()
BASE_C = BASE[BASE.kind=="control"].nll.mean()

print("baseline target NLL :", BASE_T)
print("baseline control NLL:", BASE_C)



## 10. Causal layer search

This is the first real selector. Routing frequency is not used.

First smoke run tests only 4 layers. Set `LAYER_LIMIT=None` after the pipeline is confirmed.


In [ ]:

CONTROL_PENALTY = 0.75
LAYER_LIMIT = 4  # change to None for all ~39 MoE layers

def causal_metrics(scores):
    t = scores[scores.kind=="target"].nll.mean()
    c = scores[scores.kind=="control"].nll.mean()
    dt = float(t - BASE_T)
    dc = float(c - BASE_C)
    return {
        "target_delta_nll": dt,
        "control_delta_nll": dc,
        "causal_specificity": dt - CONTROL_PENALTY * max(dc, 0.0),
    }

layers = MOE_LAYERS if LAYER_LIMIT is None else MOE_LAYERS[:LAYER_LIMIT]
layer_results = []

for layer in layers:
    set_control(zero_layers=[layer])
    s = score_set(EVAL)
    clear_control()
    layer_results.append({"layer_name": layer, **causal_metrics(s)})

layer_df = pd.DataFrame(layer_results).sort_values(
    "causal_specificity", ascending=False
).reset_index(drop=True)

display(layer_df)
layer_df.to_csv(WORK/"causal_layers.csv", index=False)



## 11. Hierarchical causal group search inside one selected layer

Rather than 256 individual ablations:

- start with expert groups;
- ablate each group with top-k fixed;
- retain the strongest groups;
- split them;
- repeat to individual experts.

This is a search heuristic; final experts are validated individually.


In [ ]:

def intervention(layer, expert_ids, renormalize=False):
    set_control(
        ablate={layer: [int(x) for x in expert_ids]},
        renormalize=renormalize,
    )
    s = score_set(EVAL)
    clear_control()
    return causal_metrics(s)

def split_blocks(xs, size):
    xs = list(xs)
    return [xs[i:i+size] for i in range(0, len(xs), size)]

def hierarchical_search(layer, initial_group=64, beam=1, min_size=1):
    frontier = split_blocks(range(256), initial_group)
    history = []
    level = 0

    while frontier:
        current = []
        for group in frontier:
            m = intervention(layer, group)
            rec = {
                "level": level,
                "layer_name": layer,
                "group_size": len(group),
                "experts": group,
                **m,
            }
            history.append(rec)
            current.append(rec)

        current.sort(key=lambda r: r["causal_specificity"], reverse=True)
        keep = current[:beam]

        if all(r["group_size"] <= min_size for r in keep):
            break

        nxt = []
        for r in keep:
            g = r["experts"]
            if len(g) <= min_size:
                nxt.append(g)
            else:
                mid = len(g)//2
                nxt += [g[:mid], g[mid:]]
        frontier = [g for g in nxt if g]
        level += 1

    hist = pd.DataFrame(history)
    smallest = hist.group_size.min()
    leaves = hist[hist.group_size==smallest].sort_values(
        "causal_specificity", ascending=False
    )
    return hist, leaves

TEST_LAYER = layer_df.iloc[0].layer_name
print("searching:", TEST_LAYER)

# Smoke settings. Later use initial_group=32, beam=2 or 3.
group_hist, leaves = hierarchical_search(
    TEST_LAYER,
    initial_group=64,
    beam=1,
    min_size=1,
)

display(leaves.head())
group_hist.to_json(WORK/"group_search.json", orient="records", indent=2)


## 12. Exact individual causal validation

In [ ]:

candidate_ids = sorted({
    int(e)
    for group in leaves.experts.tolist()
    for e in group
})

rows = []
for eid in candidate_ids:
    rows.append({
        "layer_name": TEST_LAYER,
        "expert": eid,
        **intervention(TEST_LAYER, [eid]),
    })

individual = pd.DataFrame(rows).sort_values(
    "causal_specificity", ascending=False
).reset_index(drop=True)

display(individual)
individual.to_csv(WORK/"individual_causal_experts.csv", index=False)



## 13. What to run after the smoke test

For the real experiment:

1. Set `LAYER_LIMIT=None`.
2. Use the top 3–5 causally specific layers.
3. Run `hierarchical_search(layer, initial_group=32, beam=2)`.
4. Repeat with `beam=3` if results are unstable.
5. Expand `EVAL` to dozens/hundreds of matched target/control cases.
6. Repeat final candidates with `renormalize=False` and `True`.
7. Test a few candidate pairs/coalitions.
8. Separately collect routing statistics and compare **routing rank vs causal rank**.
9. Only then train the selected experts.

### If loading still fails

Do not add CPU offload.

Keep:

```python
safetensors_load_strategy="lazy"
max_parallel_loading_workers=1
swap_space=0
cpu_offload_gb=0
```

If GPU memory is the error, reduce `max_model_len` from 1536 to 1024.

The purpose is to keep XS.2 in vLLM's quantized MoE path and perform surgery only at the actual selected top-k weights.
